<a href="https://colab.research.google.com/github/eg424/MIP/blob/Simulation/Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install magpylib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 2.0 MB/s eta 0:00:00


# Magnets setup

In [69]:
import magpylib as mp
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm
from scipy.spatial.transform import Rotation as R

# Select mode: 'single' (ball), 'chain', 'square' or 'ring' arragements
#mode = "single"
mode = "chain"
#mode = "square"
#mode = "ring"

# Neodymium - N40 Magnet Properties
magnet_properties = mp.magnet.Sphere(
    polarization=(0,0,1.24), # Z-axis. Retrieved from https://www.eclipsemagnetics.com/site/assets/files/19485/ndfeb_neodymium_iron_boron-standard_ndfeb_range_datasheet_rev1.pdf
    diameter=0.001, # 1mm
)
# Define positions based on arrangement:
if mode == "single":
  positions = [(0, 0, 0)]
elif mode == "chain":
  positions = [(0, 0, 0), (0.0016, 0, 0), (0.0046, 0, 0), (0.0076, 0, 0)]
  #positions = [(0, 0, 0), (0.0026, 0, 0), (0.0066, 0, 0), (0.0106, 0, 0)] # For plotting (2mm)
elif mode == "square":
  positions = [(0, 0, 0), (0.0016, 0, 0), (0, 0.0016, 0), (0.0016, 0.0016, 0)]
  #positions = [(0, 0, 0), (0.0026, 0, 0), (0, 0.0026, 0), (0.0026, 0.0026, 0)]
elif mode == "ring":
  r_ring = 0.0016  # 1.6 mm radius
  #r_ring = 0.0026
  angles_deg = np.arange(0,360,60)
  angles_rad = np.radians(angles_deg)

  positions = [
      (r_ring * np.cos(theta), r_ring * np.sin(theta), 0.0)
      for theta in angles_rad
      ]

# Generate rotation objects
rots = []
if mode == "chain":
  # Rotate polarization of all magnets to x-axis (90 deg)
  rots = [R.from_rotvec(np.radians([0, 90, 0]), degrees=False)] * len(positions)

elif mode == "square":
  rot_bl = R.from_euler('yx', [-45, -90], degrees=True) # Bottom Left
  rot_br = R.from_euler('yx', [-45, 90], degrees=True) # Bottom Right
  rot_tl = R.from_euler('yx', [45, -90], degrees=True) # Top Left
  rot_tr = R.from_euler('yx', [45, 90], degrees=True) # Top Right
  # Store in order of positions
  rots = [rot_bl, rot_br, rot_tl, rot_tr]

elif mode == "single":
  rots = [R.identity()] # No rotation

elif mode == "ring":
  # Rotate from Z to X
  base_rot = R.from_euler('x', -90, degrees=True)
  y_angles = [0, 60, 120, 180, 240, 300]

  rots = []
  for angle in y_angles:
      # Rotate around Z-axis to distribute around the ring
      ring_rot = R.from_euler('z', angle, degrees=True)
      # Final orientation: rotate dipole from Z to X, then spin around Z to point outward
      rot = ring_rot * base_rot
      rots.append(rot)

# Create spheres
spheres = []
for pos, Robj in zip(positions, rots):
    sph = magnet_properties.copy()
    sph.position = pos
    sph.orientation = Robj
    spheres.append(sph)

# Put them in a collection
magnets_collection = mp.Collection(*spheres)

# Create a sensor for measuring the field
sensor = mp.Sensor()

# Visualise arranged units
#mp.show(magnets_collection, sensor, backend = 'plotly')

# Simulations

External MF to break interaction between modules

In [72]:
# Interactions between magnets

delta = 1e-6  # step size

# Tracking variables
F_max = 0
max_pair = (None, None)
d = 0

# Magnetic moment (= 0.00051667 A·m²)
M = magnet_properties.magnetization # [A/m]
r = magnet_properties.diameter / 2
V = 4/3 * np.pi * r**3
magnetic_moment = M * V

# Interaction forces between modules
def compute_force(source, target):
    pos_target = np.array(target.position)

    gradB = np.zeros((3, 3))

    for i in range(3):  # x, y, z
        offset = np.zeros(3)
        offset[i] = delta
        B_plus = source.getB(pos_target + offset)
        B_minus = source.getB(pos_target - offset)
        gradB[:, i] = (B_plus - B_minus) / (2 * delta)

    # F = ∇(m·B) = (∇B)^T @ m
    return gradB.T @ magnetic_moment

# Interaction torques between modules
# def compute_torque(source, target):
#     pos_target = np.array(target.position)
#     B_field = source.getB(pos_target)

#     # Magnetization vector (local)
#     M_target = target.magnetization
#     V = 4/3 * np.pi * (target.diameter / 2)**3
#     m_target_local = M_target * V

#     # Rotate magnetic moment from local to global coordinates
#     m_target_global = target.orientation.apply(m_target_local)

#     # τ = m × B
#     torque = np.cross(m_target_global, B_field)
#     return torque

# Compute forces and torques over all pairs
n = len(spheres)
print(f"{mode.upper()}'s Pairwise Magnetic Interactions:")
for i in range(n):
    for j in range(i+1, n):
        m_source = spheres[i]
        m_target = spheres[j]

        force = compute_force(m_source, m_target)
        force_mag = np.linalg.norm(force)
        # print(f"Force from m{i+1} on m{j+1} = {force_mag*1e3:.3f} mN")

        # torque = compute_torque(m_source, m_target)
        # torque_mag = np.linalg.norm(torque)
        # print(f"Torque from m{i+1} on m{j+1} = {torque_mag*1e3:.3f} µNm")

        # Track max force and distance between its magnets
        if force_mag > F_max:
            F_max = force_mag
            pos_i = np.array(m_source.position)
            pos_j = np.array(m_target.position)
            d = np.linalg.norm(pos_j - pos_i)
            max_pair = (i+1, j+1)

# Minimum external MF
T_max = F_max * d / 2 # Torque = Force * lever arm (1/2 d between their centres)
B_min = T_max / np.linalg.norm(magnetic_moment) # T = B x M -> B = T / M

print(f"Max force = {F_max*1e3:.3f} mN between magnets {max_pair[0]} and {max_pair[1]}")
print(f"Distance between them = {d*1e3:.3f} mm")
print(f"Torque to overcome = {T_max*1e6:.3f} µN·m")
print(f"Required external MF = {B_min*1e3:.3f} mT")

CHAIN's Pairwise Magnetic Interactions:
Max force = 12.220 mN between magnets 1 and 2
Distance between them = 1.600 mm
Torque to overcome = 9.776 µN·m
Required external MF = 18.921 mT


Field on A Grid

In [ ]:
# Following guide on https://magpylib.readthedocs.io/en/stable/_pages/user_guide/examples/examples_tutorial_field_computation.html#examples-tutorial-field-computation-sensors

fig, [[ax1,ax2], [ax3,ax4]] = plt.subplots(2, 2, figsize=(10, 10))

# Create a 2D grid (mm)
if mode == "single":
  # X-Z plane at Y=0
  x_vals = np.linspace(-0.005, 0.005, 100) * 1000
  z_vals = np.linspace(-0.005, 0.005, 200) * 1000
  xlim, ylim = (-5, 5), (-5, 5)
  X, Z = np.meshgrid(x_vals, z_vals)
  grid_points = np.zeros((X.size, 3))
  grid_points[:, 0] = (X.flatten() / 1000)
  grid_points[:, 2] = (Z.flatten() / 1000)

elif mode == "chain":
  # X-Y plane at Z=0
  x_vals = np.linspace(-0.005, 0.015, 100) * 1000
  y_vals = np.linspace(-0.005, 0.005, 200) * 1000
  xlim, ylim =(-5, 15), (-5, 5)
  X, Y = np.meshgrid(x_vals, y_vals)
  grid_points = np.zeros((X.size, 3))
  grid_points[:, 0] = (X.flatten() / 1000)
  grid_points[:, 1] = (Y.flatten() / 1000)
  grid_points[:, 2] = 0.0

elif mode == "square":
  # X-Y plane at Z=0
  x_vals = np.linspace(-0.005, 0.0075, 200) * 1000
  y_vals = np.linspace(-0.005, 0.0075, 200) * 1000
  xlim, ylim =(-5, 7.5),(-5, 7.5)
  X, Y = np.meshgrid(x_vals, y_vals)
  grid_points = np.zeros((X.size, 3))
  grid_points[:, 0] = (X.flatten() / 1000)
  grid_points[:, 1] = (Y.flatten() / 1000)
  grid_points[:, 2] = 0.0

elif mode == "ring":
  # X-Y plane at Z=0
  x_vals = np.linspace(-0.005, 0.005, 200) * 1000
  y_vals = np.linspace(-0.005, 0.005, 200) * 1000
  xlim, ylim =(-5, 5),(-5, 5)
  X, Y = np.meshgrid(x_vals, y_vals)
  grid_points = np.zeros((X.size, 3))
  grid_points[:, 0] = (X.flatten() / 1000)
  grid_points[:, 1] = (Y.flatten() / 1000)
  grid_points[:, 2] = 0.0


# Compute BHJM-fields
B = magnets_collection.getB(grid_points)
H = magnets_collection.getH(grid_points)
J = magnets_collection.getJ(grid_points)
M = magnets_collection.getM(grid_points)

# Reshape for plotting
if mode == "single":
  Bx, Bz = B[:, 0].reshape(X.shape), B[:, 2].reshape(X.shape)
  Hx, Hz = H[:, 0].reshape(X.shape), H[:, 2].reshape(X.shape)
  Jx, Jz = J[:, 0].reshape(X.shape), J[:, 2].reshape(X.shape)
  Mx, Mz = M[:, 0].reshape(X.shape), M[:, 2].reshape(X.shape)

  Bmag = np.clip(norm(B[:, [0, 2]], axis=1).reshape(X.shape), 1e-12, None)
  Hmag = np.clip(norm(H[:, [0, 2]], axis=1).reshape(X.shape), 1e-12, None)
  Jmag = norm(J[:, [0, 2]], axis=1).reshape(X.shape)
  Mmag = norm(M[:, [0, 2]], axis=1).reshape(X.shape)

  ax1.streamplot(X, Z, Bx, Bz, color=np.log(Bmag), cmap="spring_r")
  ax2.streamplot(X, Z, Hx, Hz, color=np.log(Hmag), cmap="winter_r")
  ax3.streamplot(X, Z, Jx, Jz, color=Jmag, cmap="summer_r")
  ax4.streamplot(X, Z, Mx, Mz, color=Mmag, cmap="autumn_r")


else:
  Bx, By = B[:, 0].reshape(X.shape), B[:, 1].reshape(X.shape)
  Hx, Hy = H[:, 0].reshape(X.shape), H[:, 1].reshape(X.shape)
  Jx, Jy = J[:, 0].reshape(X.shape), J[:, 1].reshape(X.shape)
  Mx, My = M[:, 0].reshape(X.shape), M[:, 1].reshape(X.shape)

  Bmag = np.clip(norm(B[:, [0, 1]], axis=1).reshape(X.shape), 1e-12, None)
  Hmag = np.clip(norm(H[:, [0, 1]], axis=1).reshape(X.shape), 1e-12, None)
  Jmag = norm(J[:, [0, 1]], axis=1).reshape(X.shape)
  Mmag = norm(M[:, [0, 1]], axis=1).reshape(X.shape)

  ax1.streamplot(X, Y, Bx, By, color=np.log(Bmag), cmap="spring_r")
  ax2.streamplot(X, Y, Hx, Hy, color=np.log(Hmag), cmap="winter_r")
  ax3.streamplot(X, Y, Jx, Jy, color=Jmag, cmap="summer_r")
  ax4.streamplot(X, Y, Mx, My, color=Mmag, cmap="autumn_r")


titles = ["B-Field", "H-Field", "J-Field", "M-Field"]
axes = [ax1, ax2, ax3, ax4]

for ax, title in zip(axes, titles):
  ax.set_title(title)
  ax.set_xlabel("x [mm]")
  if mode == "single":
      ax.set_ylabel("z [mm]")
  else:
      ax.set_ylabel("y [mm]")
  ax.set_aspect("equal")
  ax.set_xlim(xlim)
  ax.set_ylim(ylim)

  # Add sphere outlines and N/S markers
  for s in spheres:
      center_mm = np.array(s.position) * 1000
      ts = np.linspace(0, 2*np.pi, 100)
      if mode == "single":
        ax.plot(center_mm[0] + np.cos(ts), center_mm[2] + np.sin(ts), 'k--', lw=0.8)
      else:
        ax.plot(center_mm[0] + np.cos(ts), center_mm[1] + np.sin(ts), 'k--', lw=0.8)


      Rmat = s.orientation.as_matrix()
      z_world = Rmat @ np.array([0, 0, 1])
      n_pos = center_mm[:2] + z_world[:2] * 1.2
      s_pos = center_mm[:2] - z_world[:2] * 1.2

      if mode == "single":
          n_pos = center_mm[[0, 2]] + z_world[[0, 2]] * 1.2
          s_pos = center_mm[[0, 2]] - z_world[[0, 2]] * 1.2
          ax.text(*n_pos, 'N', color='red', fontsize=10, ha='center', va='center', fontweight='bold')
          ax.text(*s_pos, 'S', color='blue', fontsize=10, ha='center', va='center', fontweight='bold')
      else:
          ax.text(*n_pos, 'N', color='red', fontsize=10, ha='center', va='center', fontweight='bold')
          ax.text(*s_pos, 'S', color='blue', fontsize=10, ha='center', va='center', fontweight='bold')


plt.tight_layout()
plt.show()